# 日频 Top K 期货轮动组合回测（固定时点排名）

本 Notebook 是独立的**日频版本**，不会覆盖原小时版的回测输出。

## 研究逻辑

1. 第一层策略由文华 WT8 产生，分别放在 `data/日Buy_And_Hold/`、`data/日SAR多头/`、`data/日MACD多头/`。
2. 每个交易日固定在 `REBALANCE_TIME` 进行一次全市场排名。排名使用相应第一层策略的历史权益收益率：

   \(R_{i,t}=E_{i,t}/E_{i,t-5}-1\)

   当前和历史权益都只使用排名截止时点已经可获得的记录。
3. 每套第一层策略独立生成一份 Top K 榜单；进入 Top K 的品种买入一手，持仓仍在 Top K 则继续持有，跌出 Top K 后退出。
4. 为保证对照实验公平，三套组合的**实际交易价格统一使用 `PRICE_SOURCE_STRATEGY` 指定的目录**（默认 `日Buy_And_Hold`）；不同策略只改变排名信号，不改变价格输入。
5. 排名发生在固定时点，委托在该品种排名时点之后的下一条可用日频记录尝试成交。由于 WT8 明细当前只保证有一个 `价格` 字段，而非完整 OHLC/盘口，本版使用下一条记录的 `价格` 作为成交价格代理，不等同于已验证的次日开盘价。

## 重要限制

- 默认排名时点为 `15:00`，请根据新导出的日频 CSV 的 `时间` 列确认：时间戳确实表示该条日频记录可用的时点。如果日线记录只标成 `00:00:00`，必须确认它不是把当天收盘数据错误地标成日初时间，否则可能产生时间前视。
- 默认使用同一自然日期、且时间不晚于排名截止时点的第一层权益快照参与当日排名；缺当日快照或历史不足 5 个有效日度快照的品种不会进入当日新排名。已有持仓在缺评分日不会仅因评分缺失自动退出。
- 主连切换时不会把旧、新主连价格差直接计入普通盈亏，但旧合约平仓、新合约开仓、换月价差和相应费用没有被完整重建。实际手续费按可配置的统一比例模拟，默认为 0，正式评估前需要配置真实成本。
- 这是历史数据回测，不是实盘结果。请先运行数据审计单元，重点确认时间戳和日线时间语义。

In [ ]:
from collections import defaultdict, Counter
from datetime import datetime, date, time, timedelta
from bisect import bisect_right
from pathlib import Path
from statistics import median
from typing import NamedTuple
import csv
import math
import re

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.ticker import FuncFormatter
from IPython.display import display

# ============================================================
# 配置：如果目录名称不同，只需要修改 folder 字段
# ============================================================
STRATEGY_CONFIG = {
    "long_hold": {
        "folder": "日Buy_And_Hold",
        "name": "日 Buy & Hold（五日收益榜）",
        "color": "#187a63",
    },
    "sar_long": {
        "folder": "日SAR多头",
        "name": "日 SAR 多头（五日收益榜）",
        "color": "#3568c8",
    },
    "macd_long": {
        "folder": "日MACD多头",
        "name": "日 MACD 多头（五日收益榜）",
        "color": "#d17c20",
    },
}

INITIAL_CAPITAL = 500_000.0
TOP_K = 10
LOOKBACK_TRADING_DAYS = 5
REBALANCE_TIME = "15:00"       # 每个交易日固定排名时点，可调整
START_DATE = "2024-01-02"       # 包含该日期；设为 None 表示使用全部历史
END_DATE = None                 # 包含该日期；设为 None 表示使用全部可用数据
PRICE_SOURCE_STRATEGY = "long_hold"  # 三种组合统一使用该目录中的合约与价格
ONE_WAY_FEE_RATE = 0.0          # 按成交名义金额计的单边比例费率；0 表示不计费
UNIVERSE_POLICY = "common"      # 当前推荐 common，保证三组比较同一品种池
MAX_SCORE_AGE_HOURS = 24.0       # 当日权益快照距排名时点超过该时长则不参与排名
MARGIN_BUFFER = 0.0              # 可设置 0.02 表示额外保留权益的 2% 作为缓冲
MANUAL_MULTIPLIERS = {}          # 少数品种无法反推乘数时可手工补充，如 {"沪铜主连": 5}
MANUAL_MARGIN_RATES = {}         # 可选手工保证金率，如 {"沪铜主连": 0.12}

if not STRATEGY_CONFIG:
    raise ValueError("STRATEGY_CONFIG 不能为空。")
if PRICE_SOURCE_STRATEGY not in STRATEGY_CONFIG:
    raise ValueError("PRICE_SOURCE_STRATEGY 必须是 STRATEGY_CONFIG 中的策略 ID。")
if UNIVERSE_POLICY not in {"common", "per_strategy"}:
    raise ValueError('UNIVERSE_POLICY 只能是 "common" 或 "per_strategy"。')
if TOP_K < 1 or LOOKBACK_TRADING_DAYS < 1:
    raise ValueError("TOP_K 和 LOOKBACK_TRADING_DAYS 必须为正整数。")
if INITIAL_CAPITAL <= 0 or ONE_WAY_FEE_RATE < 0:
    raise ValueError("初始资金必须大于 0，手续费率不能小于 0。")
if MAX_SCORE_AGE_HOURS <= 0 or MARGIN_BUFFER < 0 or MARGIN_BUFFER >= 1:
    raise ValueError("评分最大时效必须大于 0，MARGIN_BUFFER 需在 [0, 1) 内。")
REBALANCE_TIME_OBJ = time.fromisoformat(REBALANCE_TIME)
START_DAY = date.fromisoformat(START_DATE) if START_DATE else None
END_DAY = date.fromisoformat(END_DATE) if END_DATE else None

# 从当前目录向上寻找项目根目录；根目录应包含 data/。
_CWD = Path.cwd().resolve()
_CANDIDATE_ROOTS = (_CWD, *_CWD.parents)
BASE_DIR = next((p for p in _CANDIDATE_ROOTS if (p / "data").is_dir() and (p / "notebooks").is_dir()), None)
if BASE_DIR is None:
    BASE_DIR = next((p for p in _CANDIDATE_ROOTS if (p / "data").is_dir()), None)
if BASE_DIR is None:
    raise FileNotFoundError(f"从 {_CWD} 向上找不到包含 data/ 的项目目录，请从 Futures_Ranking 项目中运行。")

DATA_DIR = BASE_DIR / "data"
OUTPUT_DIR = BASE_DIR / "backtest_output_daily"  # 与小时版分开，避免覆盖
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("当前工作目录：", _CWD)
print("项目根目录：", BASE_DIR)
print("日频输入目录：", DATA_DIR)
print("日频输出目录：", OUTPUT_DIR)
print("统一日排名时点：", REBALANCE_TIME)
print("第一层策略数据目录：")
for sid, cfg in STRATEGY_CONFIG.items():
    folder = DATA_DIR / cfg["folder"]
    count = len(list(folder.glob("*.csv"))) if folder.is_dir() else 0
    print(f"  {sid}: {folder}（CSV={count}）")
    if not folder.is_dir() or count == 0:
        raise FileNotFoundError(
            f"找不到日频策略数据目录或目录为空：{folder}。请先创建并导出 WT8 日频 CSV，或修改 STRATEGY_CONFIG。"
        )
print("提示：首次运行请特别核对下一单元打印的时间戳时分分布，确认 15:00 的数据可用性语义。")


## 1. 读取 WT8 日频明细

WT8 导出文件采用与小时版相同的列布局：`时间`、`合约`、`信号`、`价格`、成交手数、成交额、手续费、可用资金、权益等。程序会按“品种名”去掉策略后缀，并将同一品种同一时间戳的重复行折叠为最后一条记录。

In [ ]:
class Bar(NamedTuple):
    timestamp: datetime
    contract: str
    signal: str
    price: float | None
    close_qty: float | None
    open_qty: float | None
    turnover: float | None
    fee: float | None
    available: float | None
    equity: float
    position: int


def parse_number(value):
    text = str(value or "").replace("\ufeff", "").strip().replace(",", "")
    if not text or text in {"---", "--", "-", "nan", "None"}:
        return None
    try:
        number = float(text)
        return number if math.isfinite(number) else None
    except ValueError:
        return None


def parse_timestamp(value):
    text = str(value or "").replace("\ufeff", "").strip()
    for fmt in (
        "%Y/%m/%d %H:%M:%S", "%Y-%m-%d %H:%M:%S",
        "%Y/%m/%d %H:%M", "%Y-%m-%d %H:%M",
        "%Y/%m/%d", "%Y-%m-%d",
    ):
        try:
            return datetime.strptime(text, fmt)
        except ValueError:
            continue
    return None


def instrument_name(path):
    name = re.sub(r"_(?:日|小时).*?_信号明细$", "", Path(path).stem)
    name = re.sub(r"_信号明细$", "", name)
    return re.sub(r"\s+", "", name)


def read_bars(path):
    raw_rows = None
    last_error = None
    for encoding in ("gb18030", "gbk", "utf-8-sig"):
        try:
            with Path(path).open("r", encoding=encoding, errors="strict", newline="") as handle:
                reader = csv.reader(handle)
                next(reader, None)  # 标题行
                raw_rows = list(reader)
            break
        except UnicodeDecodeError as exc:
            last_error = exc
    if raw_rows is None:
        raise UnicodeError(f"无法解码 {Path(path).name}: {last_error}")

    bars = []
    position = 0
    for raw in raw_rows:
        # WT8 表头“手数(平,开)”会拆成两个列，尾部也可能存在空字段。
        if len(raw) < 15:
            continue
        ts = parse_timestamp(raw[1])
        price = parse_number(raw[7])
        equity = parse_number(raw[14])
        if ts is None or equity is None:
            continue  # 汇总行或无效行
        signal = str(raw[3] or "").strip().upper()
        close_qty = parse_number(raw[8]) if len(raw) > 8 else None
        open_qty = parse_number(raw[9]) if len(raw) > 9 else None
        turnover = parse_number(raw[10]) if len(raw) > 10 else None
        fee = parse_number(raw[11]) if len(raw) > 11 else None
        available = parse_number(raw[13]) if len(raw) > 13 else None

        if signal == "BK":
            position = 1
        elif signal == "SP":
            position = 0
        elif signal in {"SK", "BP"}:
            position = 0  # 本 Notebook 当前假设第一层及组合均为多头样本

        bars.append(Bar(
            timestamp=ts,
            contract=str(raw[2] or "").strip(),
            signal=signal,
            price=price,
            close_qty=close_qty,
            open_qty=open_qty,
            turnover=turnover,
            fee=fee,
            available=available,
            equity=equity,
            position=position,
        ))

    bars.sort(key=lambda b: b.timestamp)
    return bars


def collapse_same_timestamp(rows):
    """同一时间戳保留排序后最后一条，返回(折叠后数据, 删除条数)。"""
    out = []
    for bar in rows:
        if out and out[-1].timestamp == bar.timestamp:
            out[-1] = bar
        else:
            out.append(bar)
    return out, len(rows) - len(out)


DATA = {}
FILE_AUDIT = []
FILE_COUNTS = {}
for sid, cfg in STRATEGY_CONFIG.items():
    folder = DATA_DIR / cfg["folder"]
    mapping = {}
    raw_count = 0
    for path in sorted(folder.glob("*.csv")):
        name = instrument_name(path)
        raw = read_bars(path)
        bars, duplicate_count = collapse_same_timestamp(raw)
        if name in mapping:
            raise ValueError(f"同一目录存在多个文件被归一化为品种 {name!r}：{path.name}")
        if not bars:
            continue
        mapping[name] = bars
        raw_count += len(raw)
        time_counts = Counter(b.timestamp.strftime("%H:%M") for b in bars)
        per_date_counts = Counter(b.timestamp.date() for b in bars)
        FILE_AUDIT.append({
            "策略ID": sid,
            "品种": name,
            "解析保留记录数": len(raw),
            "去重后记录数": len(bars),
            "同时间戳删除数": duplicate_count,
            "起始时间": bars[0].timestamp,
            "结束时间": bars[-1].timestamp,
            "每自然日最大记录数": max(per_date_counts.values()) if per_date_counts else 0,
            "常见时分": ", ".join(f"{t}({n})" for t, n in time_counts.most_common(3)),
        })
    DATA[sid] = mapping
    FILE_COUNTS[sid] = {"CSV 文件数": len(list(folder.glob("*.csv"))), "有有效记录品种数": len(mapping), "去重前有效记录总数": raw_count, "去重后记录总数": sum(len(rows) for rows in mapping.values())}

print("已读取策略数据：")
for sid, counts in FILE_COUNTS.items():
    print(sid, counts)

file_audit_df = pd.DataFrame(FILE_AUDIT)
print("\n各目录时间戳时分分布（前 20 行）：")
display(file_audit_df.head(20))

# 跨策略核对：区分“精确时间戳不一致”和“交易日期集合不一致”。
timestamp_audit_rows = []
all_names_for_audit = sorted(set().union(*(set(mapping) for mapping in DATA.values())))
for name in all_names_for_audit:
    ts_sets = {sid: {b.timestamp for b in DATA[sid].get(name, [])} for sid in STRATEGY_CONFIG}
    date_sets = {sid: {ts.date() for ts in ts_sets[sid]} for sid in STRATEGY_CONFIG}
    missing_sids = [sid for sid in STRATEGY_CONFIG if name not in DATA[sid]]
    if not missing_sids:
        common_ts = set.intersection(*(ts_sets[sid] for sid in STRATEGY_CONFIG))
        union_ts = set.union(*(ts_sets[sid] for sid in STRATEGY_CONFIG))
        common_dates = set.intersection(*(date_sets[sid] for sid in STRATEGY_CONFIG))
        union_dates = set.union(*(date_sets[sid] for sid in STRATEGY_CONFIG))
        same_ts = all(ts_sets[sid] == ts_sets[next(iter(STRATEGY_CONFIG))] for sid in STRATEGY_CONFIG)
        same_dates = all(date_sets[sid] == date_sets[next(iter(STRATEGY_CONFIG))] for sid in STRATEGY_CONFIG)
    else:
        common_ts, union_ts, common_dates, union_dates = set(), set(), set(), set()
        same_ts, same_dates = False, False
    row = {
        "品种": name,
        "缺失策略目录": ", ".join(missing_sids),
        "时间戳完全一致": same_ts,
        "交易日期集合一致": same_dates,
        "共同时间戳数": len(common_ts),
        "时间戳并集数": len(union_ts),
        "共同交易日期数": len(common_dates),
        "交易日期并集数": len(union_dates),
    }
    for sid in STRATEGY_CONFIG:
        row[f"{sid}_唯一时间数"] = len(ts_sets[sid])
        row[f"{sid}_唯一日期数"] = len(date_sets[sid])
        row[f"{sid}_独有日期数"] = len(date_sets[sid] - common_dates) if not missing_sids else None
    timestamp_audit_rows.append(row)

daily_timestamp_audit_df = pd.DataFrame(timestamp_audit_rows)
matched_audit = daily_timestamp_audit_df[daily_timestamp_audit_df["缺失策略目录"].eq("")]
print("\n三套日频数据时间戳审计：")
print("涉及品种数：", len(daily_timestamp_audit_df))
print("三个策略文件齐全：", len(matched_audit))
print("精确时间戳完全一致：", int(matched_audit["时间戳完全一致"].sum()))
print("交易日期集合一致：", int(matched_audit["交易日期集合一致"].sum()))
print("缺少策略目录文件：", len(daily_timestamp_audit_df) - len(matched_audit))
print("不一致样例（最多 20 行）：")
display(daily_timestamp_audit_df[
    (~daily_timestamp_audit_df["时间戳完全一致"])
    | (~daily_timestamp_audit_df["交易日期集合一致"])
    | (daily_timestamp_audit_df["缺失策略目录"] != "")
].head(20))

# 统一交易价格使用一个固定来源，防止三个策略在不同价格时间序列上比较。
PRICE_DATA = DATA[PRICE_SOURCE_STRATEGY]
if UNIVERSE_POLICY == "common":
    universe = set.intersection(*(set(DATA[sid]) for sid in STRATEGY_CONFIG))
    universe &= set(PRICE_DATA)
else:
    # 即使选 per_strategy，也必须存在于统一价格源中。
    universe = set(PRICE_DATA) & set.union(*(set(DATA[sid]) for sid in STRATEGY_CONFIG))

INSTRUMENTS = sorted(universe)
if not INSTRUMENTS:
    raise ValueError("没有可用的共同品种。请检查文件名和三个日频数据目录。")
print(f"\n可回测品种数：{len(INSTRUMENTS)}；统一交易价格来源：{STRATEGY_CONFIG[PRICE_SOURCE_STRATEGY]['folder']}")

# 诊断日线是否真的近似“一品种一交易日一条记录”。
max_rows_per_date = file_audit_df["每自然日最大记录数"].max() if not file_audit_df.empty else 0
if max_rows_per_date and max_rows_per_date > 1:
    print("警告：部分文件同一自然日内有多个不同时间戳；排名会用截止时点之前当日最后一条记录，成交会按下一条原始记录。请检查它们是否确实是日线导出。")
all_clock_times = Counter()
for name in INSTRUMENTS:
    for bar in PRICE_DATA[name]:
        if START_DAY is None or bar.timestamp.date() >= START_DAY:
            if END_DAY is None or bar.timestamp.date() <= END_DAY:
                all_clock_times[bar.timestamp.strftime("%H:%M")] += 1
print("\n统一价格源的常见记录时分：")
print(all_clock_times.most_common(12))
if all_clock_times and all_clock_times.most_common(1)[0][0] == "00:00":
    print("警告：价格记录大量标记为 00:00。请确认该时间代表日线实际完成/可用时间，而不是日线收盘数据的日期标签；否则固定时点回测可能存在时间前视。")


## 2. 估算合约乘数与保证金率

合约乘数从 WT8 的实际成交额估算：`成交额 ÷ (价格 × 成交手数)`。保证金率从持仓期间的 `权益 - 可用资金` 估算，并按时间保存；回测时只使用不晚于当前时点的最近历史保证金率。无法反推的品种可在配置区填写 `MANUAL_MULTIPLIERS` / `MANUAL_MARGIN_RATES`。

In [ ]:
MULTIPLIER_SAMPLES = defaultdict(list)
MARGIN_SAMPLES = defaultdict(lambda: defaultdict(list))

for sid, mapping in DATA.items():
    for name in INSTRUMENTS:
        bars = mapping.get(name, [])
        for bar in bars:
            qty = bar.open_qty if bar.signal == "BK" else bar.close_qty if bar.signal == "SP" else None
            if bar.signal in {"BK", "SP"} and bar.price and bar.turnover and qty:
                sample = bar.turnover / (bar.price * qty)
                if math.isfinite(sample) and sample > 0:
                    MULTIPLIER_SAMPLES[name].append(sample)
            if bar.position and bar.price and bar.equity is not None and bar.available is not None:
                margin_amount = bar.equity - bar.available
                if margin_amount > 0:
                    # 先用反推乘数；若暂时没有，则在下方乘数确定后再统一计算。
                    MARGIN_SAMPLES[name][bar.timestamp].append((margin_amount, bar.price))

CONTRACT_MULTIPLIER = {}
MULTIPLIER_DIAGNOSTICS = []
for name in INSTRUMENTS:
    if name in MANUAL_MULTIPLIERS:
        mult = float(MANUAL_MULTIPLIERS[name])
        samples = MULTIPLIER_SAMPLES.get(name, [])
    else:
        samples = MULTIPLIER_SAMPLES.get(name, [])
        if not samples:
            continue
        center = median(samples)
        nearest = round(center)
        mult = float(nearest) if nearest > 0 and abs(center - nearest) <= max(0.01, center * 0.005) else center
    if mult <= 0:
        continue
    CONTRACT_MULTIPLIER[name] = mult
    disp = max((abs(x - mult) / mult for x in samples), default=0.0)
    MULTIPLIER_DIAGNOSTICS.append({
        "品种": name,
        "合约乘数": mult,
        "用于估算的成交样本数": len(samples),
        "乘数样本最大相对偏差": disp,
        "是否手动配置": name in MANUAL_MULTIPLIERS,
    })

missing_multiplier = sorted(set(INSTRUMENTS) - set(CONTRACT_MULTIPLIER))
if missing_multiplier:
    raise ValueError(
        "以下品种无法从 WT8 成交明细反推合约乘数。请在 MANUAL_MULTIPLIERS 中补充后重跑："
        + ", ".join(missing_multiplier[:30])
    )

MARGIN_HISTORY = {}
for name in INSTRUMENTS:
    timeline = []
    for ts, observations in sorted(MARGIN_SAMPLES[name].items()):
        mult = CONTRACT_MULTIPLIER[name]
        rates = []
        for margin_amount, price in observations:
            notional = price * mult
            rate = margin_amount / notional if notional > 0 else None
            if rate is not None and 0.001 <= rate <= 0.50:
                rates.append(rate)
        if rates:
            timeline.append((ts, median(rates)))
    if not timeline and name in MANUAL_MARGIN_RATES:
        timeline = [(datetime.min, float(MANUAL_MARGIN_RATES[name]))]
    MARGIN_HISTORY[name] = timeline

MARGIN_TIMES = {name: [ts for ts, _ in timeline] for name, timeline in MARGIN_HISTORY.items()}

def margin_rate_at(name, timestamp):
    timeline = MARGIN_HISTORY.get(name, [])
    if not timeline:
        return None
    idx = bisect_right(MARGIN_TIMES[name], timestamp) - 1
    return timeline[idx][1] if idx >= 0 else None

multiplier_df = pd.DataFrame(MULTIPLIER_DIAGNOSTICS).sort_values("品种")
print("合约乘数估算表（前 30 行）：")
display(multiplier_df.head(30))
no_margin = [name for name in INSTRUMENTS if not MARGIN_HISTORY.get(name)]
print("没有任何可用保证金率历史的品种数：", len(no_margin))
if no_margin:
    print("这些品种开仓将被拒绝，除非在 MANUAL_MARGIN_RATES 中配置：", no_margin[:30])


## 3. 构建每日固定时点的评分快照和统一排名日历

每个品种只用 `timestamp <= 当日固定排名时点` 的记录。若该品种当天没有符合截止时间的权益记录，则不生成当日评分。历史基准使用该品种前 `LOOKBACK_TRADING_DAYS` 个可用日度快照；历史不足时不参与排名。

In [ ]:
def in_date_range(ts):
    d = ts.date()
    return (START_DAY is None or d >= START_DAY) and (END_DAY is None or d <= END_DAY)


def daily_snapshots(rows):
    """按自然日期，保留 REBALANCE_TIME 之前最后一条策略权益记录。"""
    by_date = {}
    for bar in rows:
        if bar.timestamp.time() > REBALANCE_TIME_OBJ:
            continue
        d = bar.timestamp.date()
        if d not in by_date or bar.timestamp > by_date[d].timestamp:
            by_date[d] = bar
    return dict(sorted(by_date.items()))

SCORE_SNAPSHOTS = {}
for sid, mapping in DATA.items():
    SCORE_SNAPSHOTS[sid] = {}
    for name in INSTRUMENTS:
        rows = mapping.get(name, [])
        SCORE_SNAPSHOTS[sid][name] = daily_snapshots(rows)

# 用统一价格源构建日排名日历；每个日期最多触发一次排名。
RANKING_DATES = set()
for name in INSTRUMENTS:
    for bar in PRICE_DATA.get(name, []):
        if in_date_range(bar.timestamp) and bar.timestamp.time() <= REBALANCE_TIME_OBJ:
            RANKING_DATES.add(bar.timestamp.date())
RANKING_DATES = sorted(RANKING_DATES)
if not RANKING_DATES:
    raise ValueError("在选择的日期范围和固定排名时点内，没有可用的排名日期。请检查 REBALANCE_TIME、START_DATE、END_DATE 和日频时间戳。")

RANKING_TIMES = [datetime.combine(d, REBALANCE_TIME_OBJ) for d in RANKING_DATES]
START_TS = datetime.combine(START_DAY, time.min) if START_DAY else min(
    bar.timestamp for name in INSTRUMENTS for bar in PRICE_DATA[name]
)
if END_DAY:
    END_TS = datetime.combine(END_DAY, time.max)
else:
    END_TS = max(bar.timestamp for name in INSTRUMENTS for bar in PRICE_DATA[name])

# 共用可交易价格时间序列，并限定在回测区间。
MARKET_BARS = {}
MARKET_TIMES = {}
MARKET_INDEX_BY_TIME = defaultdict(list)
MARKET_LAST_INDEX = {}
for name in INSTRUMENTS:
    rows, _ = collapse_same_timestamp(PRICE_DATA[name])
    rows = [b for b in rows if b.timestamp <= END_TS]
    MARKET_BARS[name] = rows
    MARKET_TIMES[name] = [b.timestamp for b in rows]
    if rows:
        MARKET_LAST_INDEX[name] = len(rows) - 1
        for idx, bar in enumerate(rows):
            if START_TS <= bar.timestamp <= END_TS:
                MARKET_INDEX_BY_TIME[bar.timestamp].append((name, idx, bar))

print("统一排名日期数：", len(RANKING_DATES))
print("排名日期前 5 个：", RANKING_DATES[:5])
print("排名日期最后 5 个：", RANKING_DATES[-5:])
print("排名时点：", REBALANCE_TIME)
print("可交易价格源：", STRATEGY_CONFIG[PRICE_SOURCE_STRATEGY]["folder"])

# 显示每套第一层策略在最近几个排名日有多少品种具备当日权益快照。
availability_rows = []
for sid in STRATEGY_CONFIG:
    snapshots = SCORE_SNAPSHOTS[sid]
    for d in RANKING_DATES[-10:]:
        count = sum(d in snapshots.get(name, {}) for name in INSTRUMENTS)
        availability_rows.append({"策略": sid, "排名日期": d, "当日可用权益快照品种数": count, "总品种数": len(INSTRUMENTS)})
availability_df = pd.DataFrame(availability_rows)
display(availability_df.tail(30))


## 4. 日频 Top K 组合回测

每套第一层策略独立排名。所有组合共用 `MARKET_BARS` 的市场价格序列、合约乘数与保证金率估计；每次排名只产生目标仓位变化，实际委托在该品种排名时点之后的下一条可用价格记录尝试成交。

In [ ]:
def make_daily_scores(strategy_id, rank_date, rank_timestamp):
    """返回该策略当日有效评分，严格使用截止排名时点前的权益数据。"""
    scores = []
    snapshots_by_instrument = SCORE_SNAPSHOTS[strategy_id]
    for name in INSTRUMENTS:
        snaps = snapshots_by_instrument.get(name, {})
        current = snaps.get(rank_date)
        if current is None or current.timestamp > rank_timestamp:
            continue
        age_hours = (rank_timestamp - current.timestamp).total_seconds() / 3600.0
        if age_hours < 0 or age_hours > MAX_SCORE_AGE_HOURS:
            continue
        valid_dates = [d for d in snaps.keys() if d <= rank_date]
        if not valid_dates or valid_dates[-1] != rank_date:
            continue
        current_idx = len(valid_dates) - 1
        if current_idx < LOOKBACK_TRADING_DAYS:
            continue
        base_date = valid_dates[current_idx - LOOKBACK_TRADING_DAYS]
        base = snaps[base_date]
        if base.equity <= 0 or not math.isfinite(base.equity):
            continue
        score = current.equity / base.equity - 1.0
        if not math.isfinite(score):
            continue
        # 没有回测区间内后续交易价格的品种不能成为有效开仓候选。
        next_idx = bisect_right(MARKET_TIMES[name], rank_timestamp)
        if next_idx >= len(MARKET_BARS[name]) - 1 or MARKET_BARS[name][next_idx].timestamp > END_TS:
            # 最后一个可用价格点没有后续数据，不允许在样本尾部刚开仓就立即强平。
            can_trade_next = False
        else:
            can_trade_next = True
        scores.append({
            "instrument": name,
            "score": score,
            "source_timestamp": current.timestamp,
            "base_timestamp": base.timestamp,
            "score_age_hours": age_hours,
            "lookback_calendar_days": (rank_date - base_date).days,
            "can_trade_next": can_trade_next,
        })
    scores.sort(key=lambda x: (-x["score"], x["instrument"]))
    for rank, item in enumerate(scores, 1):
        item["rank"] = rank
    # 只在能于样本区间内执行的候选中取 Top K；不可交易品种不占名额。
    eligible_for_selection = [item for item in scores if item["can_trade_next"]]
    selected_names = {item["instrument"] for item in eligible_for_selection[:TOP_K]}
    for item in scores:
        item["selected_top_k"] = item["instrument"] in selected_names
    return scores


def run_backtest(strategy_id):
    bars_by_instrument = {name: MARKET_BARS[name] for name in INSTRUMENTS if MARKET_BARS.get(name)}
    times_by_instrument = {name: MARKET_TIMES[name] for name in bars_by_instrument}
    data_ended = set()
    positions = {}
    pending = {}
    trades = []
    rejections = []
    ranking_history = []
    position_snapshots = []
    equity_curve = [{
        "timestamp": START_TS,
        "equity": INITIAL_CAPITAL,
        "gross_pnl": 0.0,
        "fees": 0.0,
        "margin_used": 0.0,
        "available": INITIAL_CAPITAL,
        "positions": 0,
        "is_ranking_tick": False,
    }]
    realized_gross_pnl = 0.0
    fees_paid = 0.0
    roll_count = 0
    rank_update_count = 0
    rank_by_name = {}
    score_by_name = {}
    source_time_by_name = {}
    rank_event_set = set(RANKING_TIMES)
    max_positions_observed = 0
    min_available_observed = INITIAL_CAPITAL
    max_margin_observed = 0.0

    def portfolio_gross_pnl():
        return realized_gross_pnl + sum(pos["gross_pnl"] for pos in positions.values())

    def portfolio_equity():
        return INITIAL_CAPITAL + portfolio_gross_pnl() - fees_paid

    def portfolio_margin(timestamp):
        total = 0.0
        for name, pos in positions.items():
            rate = margin_rate_at(name, timestamp)
            if rate is None:
                rate = pos.get("last_margin_rate")
            if rate is not None and pos.get("last_price") is not None:
                total += pos["last_price"] * CONTRACT_MULTIPLIER[name] * rate
        return total

    def close_at(name, timestamp, price, reason):
        nonlocal fees_paid, realized_gross_pnl
        pos = positions.pop(name)
        mult = CONTRACT_MULTIPLIER[name]
        exit_fee = price * mult * ONE_WAY_FEE_RATE
        fees_paid += exit_fee
        realized_gross_pnl += pos["gross_pnl"]
        net = pos["gross_pnl"] - pos["entry_fee"] - exit_fee
        trades.append({
            "strategy_id": strategy_id,
            "strategy": STRATEGY_CONFIG[strategy_id]["name"],
            "instrument": name,
            "entry_time": pos["entry_time"],
            "exit_time": timestamp,
            "entry_contract": pos["entry_contract"],
            "exit_contract": pos["contract"],
            "entry_price": pos["entry_price"],
            "exit_price": price,
            "multiplier": mult,
            "entry_notional": pos["entry_notional"],
            "entry_margin": pos["entry_margin"],
            "gross_pnl": pos["gross_pnl"],
            "fees": pos["entry_fee"] + exit_fee,
            "net_pnl": net,
            "margin_return": net / pos["entry_margin"] if pos["entry_margin"] else None,
            "holding_days": (timestamp.date() - pos["entry_time"].date()).days,
            "rolls": pos["rolls"],
            "exit_reason": reason,
        })

    def record_snapshot(timestamp, rank_date=None):
        margin_used = portfolio_margin(timestamp)
        equity = portfolio_equity()
        common = {
            "timestamp": timestamp,
            "equity": equity,
            "gross_pnl": portfolio_gross_pnl(),
            "fees": fees_paid,
            "margin_used": margin_used,
            "available": equity - margin_used,
            "position_count": len(positions),
            "ranking_date": rank_date,
        }
        equity_curve.append({**common, "positions": len(positions), "is_ranking_tick": rank_date is not None})
        if not positions:
            position_snapshots.append({**common, "instrument": "", "contract": "", "entry_time": None,
                                       "entry_price": None, "mark_price": None, "floating_pnl": 0.0,
                                       "rank": None, "five_day_return": None, "score_source_timestamp": None})
            return
        for name, pos in sorted(positions.items(), key=lambda kv: (rank_by_name.get(kv[0], TOP_K + 1), kv[0])):
            position_snapshots.append({
                **common,
                "instrument": name,
                "contract": pos["contract"],
                "entry_time": pos["entry_time"],
                "entry_price": pos["entry_price"],
                "mark_price": pos["last_price"],
                "floating_pnl": pos["gross_pnl"] - pos["entry_fee"],
                "rank": rank_by_name.get(name),
                "five_day_return": score_by_name.get(name),
                "score_source_timestamp": source_time_by_name.get(name),
            })

    # 将所有价格时点和每日固定排名时点合并成事件时钟。
    event_times = sorted(
        {ts for ts in MARKET_INDEX_BY_TIME if START_TS <= ts <= END_TS}
        | {ts for ts in rank_event_set if START_TS <= ts <= END_TS}
    )
    for timestamp in event_times:
        rows_now = MARKET_INDEX_BY_TIME.get(timestamp, [])
        rows_now_by_name = {name: (idx, bar) for name, idx, bar in rows_now}

        # 1) 先按当下可用价格更新已持仓品种的浮动盈亏；主连换月不把跨合约价差当作普通盈亏。
        for name, (idx, bar) in rows_now_by_name.items():
            if name in positions and bar.price is not None and bar.price > 0:
                pos = positions[name]
                old_contract = pos["contract"]
                if old_contract and bar.contract and old_contract != bar.contract:
                    pos["rolls"] += 1
                    roll_count += 1
                elif pos["last_price"] is not None:
                    pos["gross_pnl"] += (bar.price - pos["last_price"]) * CONTRACT_MULTIPLIER[name]
                pos["last_price"] = bar.price
                pos["contract"] = bar.contract or old_contract
                rate = margin_rate_at(name, timestamp)
                if rate is not None:
                    pos["last_margin_rate"] = rate

        # 2) 到达下一条价格记录时，优先平仓，再开仓。
        due = []
        for name, order in list(pending.items()):
            if order["due_timestamp"] == timestamp and name in rows_now_by_name:
                due.append((name, order, rows_now_by_name[name][1]))
        due.sort(key=lambda item: (0 if item[1]["action"] == "exit" else 1, item[0]))
        for name, order, bar in due:
            if pending.get(name) is not order:
                continue
            if bar.price is None or bar.price <= 0:
                next_times = times_by_instrument.get(name, [])
                next_idx = bisect_right(next_times, timestamp)
                if next_idx < len(next_times) and next_times[next_idx] <= END_TS:
                    order["due_timestamp"] = next_times[next_idx]
                else:
                    rejections.append({"timestamp": timestamp, "instrument": name, "reason": "执行时价格无效且无后续可用价格"})
                    pending.pop(name, None)
                continue
            action = order["action"]
            if action == "exit":
                if name in positions:
                    close_at(name, timestamp, bar.price, order.get("reason", "跌出Top K"))
                pending.pop(name, None)
            elif action == "enter":
                if name in positions:
                    pending.pop(name, None)
                    continue
                if name in data_ended:
                    pending.pop(name, None)
                    continue
                if len(positions) >= TOP_K:
                    # 满仓时将委托顺延到该品种下一条价格记录；下一次日排名可撤销它。
                    next_times = times_by_instrument.get(name, [])
                    next_idx = bisect_right(next_times, timestamp)
                    if next_idx < len(next_times) and next_times[next_idx] <= END_TS:
                        order["due_timestamp"] = next_times[next_idx]
                    else:
                        pending.pop(name, None)
                        rejections.append({"timestamp": timestamp, "instrument": name, "reason": "组合满仓且无后续可用记录"})
                    continue
                rate = margin_rate_at(name, timestamp)
                if rate is None:
                    rejections.append({"timestamp": timestamp, "instrument": name, "reason": "没有可用历史保证金率"})
                    pending.pop(name, None)
                    continue
                mult = CONTRACT_MULTIPLIER[name]
                notional = bar.price * mult
                required_margin = notional * rate
                entry_fee = notional * ONE_WAY_FEE_RATE
                eq = portfolio_equity()
                margin_now = portfolio_margin(timestamp)
                if margin_now + required_margin + entry_fee > eq * (1.0 - MARGIN_BUFFER):
                    rejections.append({"timestamp": timestamp, "instrument": name, "reason": "可用资金/保证金不足"})
                    pending.pop(name, None)
                    continue
                fees_paid += entry_fee
                positions[name] = {
                    "entry_time": timestamp,
                    "entry_contract": bar.contract,
                    "contract": bar.contract,
                    "entry_price": bar.price,
                    "last_price": bar.price,
                    "entry_notional": notional,
                    "entry_margin": required_margin,
                    "entry_fee": entry_fee,
                    "gross_pnl": 0.0,
                    "last_margin_rate": rate,
                    "rolls": 0,
                }
                pending.pop(name, None)

        # 3) 品种数据结束时在最后一条有效价格处结束该持仓，不延用陈旧价格。
        for name, (idx, bar) in rows_now_by_name.items():
            if idx >= len(bars_by_instrument[name]) - 1:
                data_ended.add(name)
                pending.pop(name, None)
                if name in positions and bar.price is not None and bar.price > 0:
                    close_at(name, timestamp, bar.price, "品种数据结束平仓")

        # 4) 仅在每日固定时点更新榜单。排名缺失不等于跌出 Top K：没有当日有效评分的持仓不因缺分自动退出。
        if timestamp in rank_event_set:
            rank_update_count += 1
            rank_date = timestamp.date()
            scores = make_daily_scores(strategy_id, rank_date, timestamp)
            valid_score_names = {row["instrument"] for row in scores}
            rank_by_name = {row["instrument"]: row["rank"] for row in scores}
            score_by_name = {row["instrument"]: row["score"] for row in scores}
            source_time_by_name = {row["instrument"]: row["source_timestamp"] for row in scores}
            target_names = {
                row["instrument"] for row in scores if row["selected_top_k"]
            }
            for row in scores:
                ranking_history.append({
                    "ranking_timestamp": timestamp,
                    "rank_date": rank_date,
                    "rank": row["rank"],
                    "instrument": row["instrument"],
                    "five_day_return": row["score"],
                    "score_source_timestamp": row["source_timestamp"],
                    "base_timestamp": row["base_timestamp"],
                    "score_age_hours": row["score_age_hours"],
                    "lookback_calendar_days": row["lookback_calendar_days"],
                    "selected_top_k": row["selected_top_k"],
                    "can_trade_next": row["can_trade_next"],
                })

            # 只更新有当日有效评分的品种；缺少评分不能自动触发离榜平仓。
            for name in sorted(set(positions) | set(pending) | target_names):
                if name not in valid_score_names:
                    continue
                wanted_action = "enter" if name in target_names and name not in positions else "exit" if name in positions and name not in target_names else None
                existing = pending.get(name)
                if existing and existing["action"] != wanted_action:
                    pending.pop(name, None)
                    existing = None
                if wanted_action and existing is None:
                    market_times = times_by_instrument.get(name, [])
                    due_idx = bisect_right(market_times, timestamp)
                    if due_idx >= len(market_times) or market_times[due_idx] > END_TS:
                        rejections.append({"timestamp": timestamp, "instrument": name, "reason": "排名时点后无下一条可交易记录"})
                        continue
                    pending[name] = {
                        "action": wanted_action,
                        "signal_timestamp": timestamp,
                        "due_timestamp": market_times[due_idx],
                        "rank": rank_by_name.get(name),
                        "reason": "跌出Top K" if wanted_action == "exit" else "进入Top K",
                    }
                elif existing and wanted_action:
                    existing["rank"] = rank_by_name.get(name)

        # 5) 保证金不足时，只在当下有真实价格记录的持仓品种中按排名靠后优先强平。
        current_equity = portfolio_equity()
        current_margin = portfolio_margin(timestamp)
        candidates = sorted(
            [name for name in positions if name in rows_now_by_name and rows_now_by_name[name][1].price is not None],
            key=lambda name: (rank_by_name.get(name, TOP_K + 1), name),
            reverse=True,
        )
        for name in candidates:
            if current_margin <= current_equity * (1.0 - MARGIN_BUFFER):
                break
            close_at(name, timestamp, positions[name]["last_price"], "保证金不足强制平仓")
            current_equity = portfolio_equity()
            current_margin = portfolio_margin(timestamp)

        max_positions_observed = max(max_positions_observed, len(positions))
        max_margin_observed = max(max_margin_observed, current_margin)
        min_available_observed = min(min_available_observed, current_equity - current_margin)

        # 日频曲线只在固定排名时点采样；实际盈亏仍会在当天原始价格记录到来时更新。
        if timestamp in rank_event_set:
            record_snapshot(timestamp, timestamp.date())

    # 最终按最后可用市场价格结束持仓，确保没有跨样本期的未平仓合约。
    last_ts = max((ts for ts in MARKET_INDEX_BY_TIME if START_TS <= ts <= END_TS), default=START_TS)
    for name in sorted(list(positions)):
        pos = positions[name]
        close_at(name, last_ts, pos["last_price"], "样本结束平仓")
    # 最后一条权益点使用最终平仓后的权益；若结束时点已有点则覆盖。
    final_point = {
        "timestamp": last_ts,
        "equity": portfolio_equity(),
        "gross_pnl": portfolio_gross_pnl(),
        "fees": fees_paid,
        "margin_used": 0.0,
        "available": portfolio_equity(),
        "positions": 0,
        "is_ranking_tick": False,
    }
    if equity_curve and equity_curve[-1]["timestamp"] == last_ts:
        equity_curve[-1].update(final_point)
    else:
        equity_curve.append(final_point)
    # 保证曲线按时间排序；同一时点可能同时是排名时点和最终时点，保留最后一条。
    dedup_curve = {}
    for point in equity_curve:
        dedup_curve[point["timestamp"]] = point
    equity_curve = [dedup_curve[t] for t in sorted(dedup_curve)]

    peak = INITIAL_CAPITAL
    max_drawdown = 0.0
    for point in equity_curve:
        peak = max(peak, point["equity"])
        dd = point["equity"] / peak - 1.0 if peak else 0.0
        max_drawdown = min(max_drawdown, dd)
    first_ts = equity_curve[0]["timestamp"]
    final_ts = equity_curve[-1]["timestamp"]
    elapsed_days = max((final_ts - first_ts).total_seconds() / 86400.0, 1.0)
    final_equity = equity_curve[-1]["equity"]
    cagr = (final_equity / INITIAL_CAPITAL) ** (365.25 / elapsed_days) - 1.0 if final_equity > 0 else -1.0
    wins = sum(t["net_pnl"] > 0 for t in trades)
    weighted_margin_seconds = 0.0
    elapsed_seconds = 0.0
    max_margin = max_margin_observed
    max_positions = max_positions_observed
    min_available = min(min_available_observed, min(p["available"] for p in equity_curve))
    for current, following in zip(equity_curve, equity_curve[1:]):
        seconds = max((following["timestamp"] - current["timestamp"]).total_seconds(), 0.0)
        weighted_margin_seconds += current["margin_used"] * seconds
        elapsed_seconds += seconds
        max_margin = max(max_margin, current["margin_used"])
        max_positions = max(max_positions, current.get("positions", 0))
    avg_margin = weighted_margin_seconds / elapsed_seconds if elapsed_seconds else 0.0
    net_pnl = final_equity - INITIAL_CAPITAL
    summary = {
        "策略": STRATEGY_CONFIG[strategy_id]["name"],
        "排名方式": f"每日 {REBALANCE_TIME}，过去 {LOOKBACK_TRADING_DAYS} 个日度权益观测收益率",
        "每日排名次数": rank_update_count,
        "回测起点": first_ts,
        "回测终点": final_ts,
        "品种数": len(INSTRUMENTS),
        "平仓笔数": len(trades),
        "净盈亏": net_pnl,
        "期末权益": final_equity,
        "组合收益率": net_pnl / INITIAL_CAPITAL,
        "年化复合收益率": cagr,
        "最大回撤": max_drawdown,
        "胜率": wins / len(trades) if trades else 0.0,
        "时间加权平均保证金": avg_margin,
        "最大保证金占用": max_margin,
        "累计收益/平均保证金": net_pnl / avg_margin if avg_margin else None,
        "最大持仓数": max_positions,
        "最低可用资金": min_available,
        "未成交/拒绝记录数": len(rejections),
        "保证金强平笔数": sum(t["exit_reason"] == "保证金不足强制平仓" for t in trades),
        "主连换月次数": sum(t["rolls"] for t in trades),
        "手续费": fees_paid,
    }
    return {
        "summary": summary,
        "equity_curve": equity_curve,
        "trades": trades,
        "position_snapshots": position_snapshots,
        "ranking_history": ranking_history,
        "rejections": rejections,
    }

RESULTS = {}
for sid in STRATEGY_CONFIG:
    print("开始日频回测：", STRATEGY_CONFIG[sid]["name"])
    RESULTS[sid] = run_backtest(sid)
    assert RESULTS[sid]["summary"]["最大持仓数"] <= TOP_K
    assert RESULTS[sid]["equity_curve"][0]["equity"] == INITIAL_CAPITAL or abs(RESULTS[sid]["equity_curve"][0]["equity"] - INITIAL_CAPITAL) < 1e-6
print("日频回测完成。提醒：当前统一单边手续费率为", ONE_WAY_FEE_RATE, "；若为 0，结果未计手续费。")


## 5. 查看策略汇总和资金曲线

三套策略使用同一市场价格序列、统一排名时点、同一资金规模和同一品种池。请先看汇总和曲线，再检查下一节导出的逐日排名与交易记录。

In [ ]:
summary_df = pd.DataFrame([RESULTS[sid]["summary"] for sid in STRATEGY_CONFIG])
summary_view = summary_df.copy()
for col in ["净盈亏", "期末权益", "时间加权平均保证金", "最大保证金占用", "最低可用资金", "手续费"]:
    if col in summary_view:
        summary_view[col] = summary_view[col].map(lambda x: f"{x:,.2f}" if pd.notna(x) else "—")
for col in ["组合收益率", "年化复合收益率", "最大回撤", "胜率", "累计收益/平均保证金"]:
    if col in summary_view:
        summary_view[col] = summary_view[col].map(lambda x: f"{x:.2%}" if pd.notna(x) else "—")
display(summary_view)

fig, ax = plt.subplots(figsize=(14, 7), constrained_layout=True)
for sid, cfg in STRATEGY_CONFIG.items():
    curve = RESULTS[sid]["equity_curve"]
    times = [p["timestamp"] for p in curve]
    equity = [p["equity"] for p in curve]
    final_return = (equity[-1] / INITIAL_CAPITAL - 1.0) * 100
    ax.plot(times, equity, label=f"{cfg['name']}  {equity[-1]/10000:,.1f}万元（{final_return:+.1f}%）", color=cfg["color"], linewidth=1.7)
ax.axhline(INITIAL_CAPITAL, linestyle="--", color="#858c98", linewidth=1.0, label=f"初始资金 {INITIAL_CAPITAL/10000:.0f}万元")
ax.set_title(f"每日 {REBALANCE_TIME} 排名：Top {TOP_K} 期货轮动组合权益曲线")
ax.set_xlabel("日期")
ax.set_ylabel("组合权益（万元）")
ax.yaxis.set_major_formatter(FuncFormatter(lambda value, _: f"{value / 10000:,.0f}"))
ax.xaxis.set_major_locator(mdates.AutoDateLocator(minticks=6, maxticks=12))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
ax.grid(axis="y", color="#e5e9ef", linewidth=0.8)
ax.spines[["top", "right"]].set_visible(False)
ax.legend(loc="best", frameon=False)
fig.savefig(OUTPUT_DIR / "daily_topk_equity_curve.png", dpi=180, bbox_inches="tight")
plt.show()
print("每日资金曲线已保存：", OUTPUT_DIR / "daily_topk_equity_curve.png")


## 6. 导出回测结果

每套第一层策略会输出一份汇总、权益曲线、交易记录、持仓快照和完整日度排名。`daily_ranking_*.csv` 可用于核验每天的 Top K 选择依据。

In [ ]:
def serializable_rows(rows):
    out = []
    for row in rows:
        item = dict(row)
        for key, value in item.items():
            if isinstance(value, (datetime, date)):
                item[key] = value.isoformat(sep=" ") if isinstance(value, datetime) else value.isoformat()
        out.append(item)
    return out


def write_csv(path, rows):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    rows = serializable_rows(rows)
    if not rows:
        path.write_text("", encoding="utf-8-sig")
        return
    with path.open("w", encoding="utf-8-sig", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=list(rows[0].keys()), extrasaction="ignore")
        writer.writeheader()
        writer.writerows(rows)

summary_export = serializable_rows([RESULTS[sid]["summary"] for sid in STRATEGY_CONFIG])
write_csv(OUTPUT_DIR / "daily_strategy_summary.csv", summary_export)
write_csv(OUTPUT_DIR / "daily_multiplier_diagnostics.csv", MULTIPLIER_DIAGNOSTICS)
write_csv(OUTPUT_DIR / "daily_input_file_audit.csv", FILE_AUDIT)
write_csv(OUTPUT_DIR / "daily_timestamp_audit.csv", timestamp_audit_rows)

for sid, result in RESULTS.items():
    write_csv(OUTPUT_DIR / f"daily_equity_curve_{sid}.csv", result["equity_curve"])
    write_csv(OUTPUT_DIR / f"daily_trades_{sid}.csv", result["trades"])
    write_csv(OUTPUT_DIR / f"daily_positions_{sid}.csv", result["position_snapshots"])
    write_csv(OUTPUT_DIR / f"daily_ranking_{sid}.csv", result["ranking_history"])
    write_csv(OUTPUT_DIR / f"daily_rejections_{sid}.csv", result["rejections"])

print("导出完成：", OUTPUT_DIR)
print("文件：daily_strategy_summary.csv, daily_timestamp_audit.csv, daily_equity_curve_*.csv,")
print("      daily_trades_*.csv, daily_positions_*.csv, daily_ranking_*.csv, daily_rejections_*.csv")


## 7. 排名核查示例

以下展示一个日期的完整有效排名（不只显示 Top K），便于核查分数来源时间和五日基准时间。修改 `AUDIT_DATE` 即可抽查其他交易日。

In [ ]:
AUDIT_DATE = RANKING_DATES[min(len(RANKING_DATES) - 1, max(0, len(RANKING_DATES) - 5))]
print("抽查日期：", AUDIT_DATE)
for sid, cfg in STRATEGY_CONFIG.items():
    rank_df = pd.DataFrame(RESULTS[sid]["ranking_history"])
    if rank_df.empty:
        print(cfg["name"], "没有可用排名记录")
        continue
    selected = rank_df[rank_df["rank_date"] == AUDIT_DATE].sort_values("rank")
    print("\n", cfg["name"])
    display(selected.head(TOP_K + 5))
